In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from io import StringIO
from datetime import datetime
import os
import json
import fileReader as fr

## Updating Userbase

In [40]:
cols = ['username', 'total_attempted', 'total_correct', 'strong_areas', 'weak_areas', 'last_5_papers_accuracy']

In [39]:
def assess(correct, counts):
    strong = []
    weak = []
    lowth = 0.3
    highth = 0.7
    
    for domain in correct.index:
        val = correct[domain] / counts[domain]
        if(val < lowth):
            weak.append(domain)
        elif(val > highth):
            strong.append(domain)
    
    return strong, weak

In [32]:
# working on it

def update(udf, row):
    cols = udf.columns
    newdf = pd.DataFrame(columns=cols)

    for i in range(udf.shape[0]):
        if(row.iloc[0]['username'] == udf.loc[i]['username']):
            rowdict = row.iloc[0].to_dict()
            udf.drop(udf[udf.username == row.iloc[0]['username']].index, inplace=True)
    newdf = pd.concat([udf, row])
    return newdf

In [33]:
qdf.iloc[0].to_dict()

{'QID': 'CH0',
 'domain': 'Solid State',
 'subdomain': 'Solid State Characteristics',
 'question': 'Solid-state is denser than the liquid and gaseous states of the same substance. Which of the following is an exception to this rule?',
 'options': "['Mercury', 'Carbon dioxide (dry ice)', 'Ice', 'NaCl']",
 'answer': 'c',
 'explaination': 'The density of ice is about 0.92 g/cm<sup>3</sup> while that of water is 1 g/cm<sup>3</sup>. Mercury has density 14.184 g/cm<sup>3</sup> as solid and 13.69 g/cm<sup>3</sup> as liquid. Carbon dioxide has density 1.56 g/cm<sup>3</sup> as solid and 1.10 g/cm<sup>3</sup> as liquid. NaCl has density 2.71 g/cm<sup>3</sup> as solid and 1.556 g/cm<sup>3</sup> in molten state. Therefore only ice has lesser density as a solid than as a liquid.'}

In [34]:
qdf.iloc[9]

QID                                                           CH9
domain                                                Solid State
subdomain                             Solid State Characteristics
question        Sulfur exists in two polymorphic forms _______...
options         ['rhombic and monoclinic', 'rhombic and tricli...
answer                                                          a
explaination    There are two polymorphous structures of sulfu...
Name: 9, dtype: object

In [35]:
def process(rr):
    full = rr.fulldf
    l5 = rr.l5df
    file = rr.filename
    name = file.split('.')[0]
    name = name.split('/')[-1]
    tq = full.shape[0]
    tc = sum(full['correct'] == full['chosen'])
    tq5 = l5.shape[0]
    true = (l5['correct'] == l5['chosen'])
    tc5 = sum(true)
    acc5 = tc5 / tq5
    l5['true'] = true
    dcorrect = l5.groupby('domain').sum()['true']
    dcounts = l5.groupby('domain').count()['correct']
    strong, weak = assess(dcorrect, dcounts)
    temp = pd.DataFrame([[name, tq, tc, strong, weak, acc5]], columns = cols)
    
    if os.path.exists('userData/users/users.csv'):
        udf = pd.read_csv('userData/users/users.csv')
    else:
        udf = pd.DataFrame(columns = cols)
        udf.to_csv('userData/users/users.csv', index=False)
    
    udf = update(udf, temp)
    udf.to_csv('userData/users/users.csv', index=False)
    return temp

In [46]:
def readFiles():
    for filename in os.listdir('userData/csvs'):
        try:
            rr = fr.reader()
            rr.read('userData/csvs/' + filename)
            process(rr)
            name = filename.split('.')[0]
            with open('userData/jsons/' + name + '.json', 'w') as outfile:
                json.dump(rr.diffdict, outfile)
            print('File read:', filename)
        except:
            print('File not read:', filename)

In [50]:
readFiles()

File read: arsed.csv
File read: wert.csv


In [51]:
t2 = pd.read_csv('userData/users/users.csv')
t2

,username,total_attempted,total_correct,strong_areas,weak_areas,last_5_papers_accuracy
0,arsed,20,1,['Electrochemistry'],"['Alcohols, Phenols & Ethers', 'Aldehydes, Ket...",0.050000
1,wert,110,2,[],"['Alcohols, Phenols & Ethers', 'Aldehydes, Ket...",0.018182


# Simulating user data (ignore)

In [5]:
qdf = pd.read_csv('IRdata2.csv').drop(['imageURL'], axis=1)
qdf["QID"] = "CH" + qdf["QID"].map(str)

In [19]:
qdf

,QID,domain,subdomain,question,options,answer,explaination
0,CH0,Solid State,Solid State Characteristics,Solid-state is denser than the liquid and gase...,"['Mercury', 'Carbon dioxide (dry ice)', 'Ice',...",c,The density of ice is about 0.92 g/cm<sup>3</s...
1,CH1,Solid State,Solid State Characteristics,Which of the following can be used to describe...,"['Heterogeneous, anisotropic', 'Homogeneous, a...",b,Homogeneity refers to uniformity in compositio...
2,CH2,Solid State,Solid State Characteristics,When a single substance can crystallize in two...,"['Polymorphous', 'Isomorphous', 'Semimorphous'...",a,Isomorphous is when two or more substances hav...
3,CH3,Solid State,Solid State Characteristics,Which of the following is an amorphous solid?,"['Quartz', 'Quartz glass', 'Graphite', 'Salt (...",b,Quartz glass does not have a perfectly ordered...
4,CH4,Solid State,Solid State Characteristics,Amorphous solids are actually supercooled liqu...,"['True', 'False']",a,Amorphous solids behave like fluids and flow v...
...,...,...,...,...,...,...,...
1801,CH1801,Everyday Life Chemistry,Cleansing Agents,Which of the following causes soap to lather?,"['Sodium carbonate', 'Sodium rosinate', 'Sodiu...",b,Rosin is a chemical compound which is a gum th...
1802,CH1802,Everyday Life Chemistry,Cleansing Agents,When a soap is dissolved in ethanol followed b...,"['Transparent soap', 'Floating soap', 'Shaving...",a,When the soap is dissolved in a solution of et...
1803,CH1803,Everyday Life Chemistry,Cleansing Agents,What is the use of trisodium phosphate in soap...,"['To make the soap act rapidly', 'To make it l...",a,Sodium carbonate and trisodium phosphate acts ...
1804,CH1804,Everyday Life Chemistry,Cleansing Agents,Dishwashing liquids are examples of ______,"['soaps', 'anionic detergents', 'cationic dete...",d,Dishwashing liquids are non-ionic detergents. ...


In [83]:
fdata = np.zeros((qdf.shape[0], 12))
fcols = ['qid', 'subdomain', 'u1time', 'u2time', 'u3time', 'u4time', 'u5time',
         'u6time', 'u7time', 'u8time', 'u9time', 'u10time']

In [84]:
fdf = pd.DataFrame(fdata, columns = fcols)

In [85]:
fdf['qid'] = qdf['QID']
fdf['subdomain'] = qdf['subdomain']

In [86]:
fdf

,qid,subdomain,u1time,u2time,u3time,u4time,u5time,u6time,u7time,u8time,u9time,u10time
0,CH0,Solid State Characteristics,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,CH1,Solid State Characteristics,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,CH2,Solid State Characteristics,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,CH3,Solid State Characteristics,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,CH4,Solid State Characteristics,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1801,CH1801,Cleansing Agents,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1802,CH1802,Cleansing Agents,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1803,CH1803,Cleansing Agents,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1804,CH1804,Cleansing Agents,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [87]:
rtimes = []

for i in range(fdf.shape[0]):
    ll = np.random.uniform(30, 90, size = (fdf.shape[0]))
    rtimes.append(ll)

In [88]:
rtimes[0]

array([89.78726557, 56.50935977, 86.22026256, ..., 71.31705576,
       59.62614225, 52.74204712])

In [89]:
for i in range(1, 11):
    col = 'u' + str(i) + 'time'
    fdf[col] = rtimes[i]

In [91]:
fdf

,qid,subdomain,u1time,u2time,u3time,u4time,u5time,u6time,u7time,u8time,u9time,u10time
0,CH0,Solid State Characteristics,32.935482,64.497045,58.351867,46.422059,88.324843,32.913939,56.035772,65.690845,42.290322,88.125968
1,CH1,Solid State Characteristics,83.036844,50.727860,74.127059,77.453555,61.703203,65.604180,36.799286,41.096888,36.387888,52.603387
2,CH2,Solid State Characteristics,32.846362,70.887914,45.610590,52.449529,44.669956,59.206373,42.343958,49.056458,75.223019,79.265696
3,CH3,Solid State Characteristics,82.136518,89.026081,58.100691,72.857707,48.529952,62.945631,72.268484,45.018586,79.076798,82.704996
4,CH4,Solid State Characteristics,81.812102,45.215877,62.204291,80.448805,54.456608,61.186478,35.945835,70.770372,33.778726,34.088833
...,...,...,...,...,...,...,...,...,...,...,...,...
1801,CH1801,Cleansing Agents,86.384309,69.199422,71.828857,70.780334,31.700318,82.838624,85.462047,39.392129,38.785222,56.395270
1802,CH1802,Cleansing Agents,72.842972,32.889731,86.375690,89.852338,55.589171,50.614545,52.101135,61.995683,51.878139,71.391776
1803,CH1803,Cleansing Agents,55.804879,76.758463,42.743565,64.158968,67.331419,47.747630,31.213373,55.661211,81.048794,48.257172
1804,CH1804,Cleansing Agents,85.182811,65.180862,68.229333,61.425714,45.986878,38.520768,84.891223,54.877733,32.322842,55.775716


In [107]:
means = fdf[fdf.columns[2:]].mean(axis=1)

In [108]:
fdf['average'] = means

In [109]:
fdf

,qid,subdomain,u1time,u2time,u3time,u4time,u5time,u6time,u7time,u8time,u9time,u10time,average
0,CH0,Solid State Characteristics,32.935482,64.497045,58.351867,46.422059,88.324843,32.913939,56.035772,65.690845,42.290322,88.125968,57.558814
1,CH1,Solid State Characteristics,83.036844,50.727860,74.127059,77.453555,61.703203,65.604180,36.799286,41.096888,36.387888,52.603387,57.954015
2,CH2,Solid State Characteristics,32.846362,70.887914,45.610590,52.449529,44.669956,59.206373,42.343958,49.056458,75.223019,79.265696,55.155985
3,CH3,Solid State Characteristics,82.136518,89.026081,58.100691,72.857707,48.529952,62.945631,72.268484,45.018586,79.076798,82.704996,69.266545
4,CH4,Solid State Characteristics,81.812102,45.215877,62.204291,80.448805,54.456608,61.186478,35.945835,70.770372,33.778726,34.088833,55.990793
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1801,CH1801,Cleansing Agents,86.384309,69.199422,71.828857,70.780334,31.700318,82.838624,85.462047,39.392129,38.785222,56.395270,63.276653
1802,CH1802,Cleansing Agents,72.842972,32.889731,86.375690,89.852338,55.589171,50.614545,52.101135,61.995683,51.878139,71.391776,62.553118
1803,CH1803,Cleansing Agents,55.804879,76.758463,42.743565,64.158968,67.331419,47.747630,31.213373,55.661211,81.048794,48.257172,57.072547
1804,CH1804,Cleansing Agents,85.182811,65.180862,68.229333,61.425714,45.986878,38.520768,84.891223,54.877733,32.322842,55.775716,59.239388


In [97]:
fdf.to_csv('userData/sampleData.csv', index=False)

In [98]:
fdf.describe()

,u1time,u2time,u3time,u4time,u5time,u6time,u7time,u8time,u9time,u10time,average
count,1806.000000,1806.000000,1806.000000,1806.000000,1806.000000,1806.000000,1806.000000,1806.000000,1806.000000,1806.000000,1806.000000
mean,60.492182,60.067327,60.238786,59.752365,59.842171,60.613210,59.972792,60.558044,59.682455,60.286055,60.150539
std,17.182476,17.202634,17.634440,17.354126,17.395357,17.543359,17.432077,17.591163,17.464788,17.230347,5.421848
min,30.004215,30.023940,30.002301,30.010342,30.001279,30.006540,30.022039,30.021251,30.013722,30.027772,43.326319
25%,45.265595,45.396205,45.332585,44.616718,44.780532,45.540679,43.912597,45.125501,44.121271,45.230461,56.376035
50%,60.431265,60.336453,60.197456,60.035682,59.838915,60.850298,60.636393,60.876779,58.901229,60.452437,60.091652
75%,75.684242,74.440444,75.922419,74.195821,74.628403,76.070042,75.063968,75.918753,75.231616,75.607458,63.940778
max,89.977217,89.989810,89.986823,89.990388,89.948453,89.928477,89.971141,89.961154,89.995486,89.968279,75.385829


In [102]:
grouped = fdf.groupby('subdomain').mean()

In [111]:
grouped.describe()

,u1time,u2time,u3time,u4time,u5time,u6time,u7time,u8time,u9time,u10time,average
count,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000
mean,60.445265,60.145315,60.393342,59.806142,59.818532,60.667231,59.981924,60.425072,59.713247,60.158295,60.155437
std,5.084146,5.316983,5.882925,5.628293,5.759923,5.993971,5.939685,5.378649,5.094980,5.843971,1.591872
min,43.713366,46.475988,40.333787,43.784617,40.856463,45.381717,45.212223,40.365321,45.748678,43.216694,56.344347
25%,57.452380,56.522764,57.134657,55.966377,56.189646,56.565650,55.960882,56.564868,55.891748,56.400041,59.115721
50%,60.985197,60.347256,60.370278,60.475785,59.999911,60.820945,59.856713,60.553392,59.955726,59.551527,60.219339
75%,63.852765,63.506527,63.747586,63.323608,63.768398,63.693089,63.607529,63.982457,63.565742,64.456981,61.041710
max,73.516580,75.647999,82.352167,75.967738,73.241053,78.956740,79.474067,74.842478,73.025737,75.898439,64.414397


In [106]:
grouped.describe()['average']

count    150.000000
mean      60.155437
std        1.591872
min       56.344347
25%       59.115721
50%       60.219339
75%       61.041710
max       64.414397
Name: average, dtype: float64

In [3]:
import cosine_ranking as cr

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Aryan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Aryan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Aryan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Aryan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [6]:
np.unique(qdf['domain'])

array(['Alcohols, Phenols & Ethers',
       'Aldehydes, Ketones & Carboxylic Acids', 'Amines', 'Biomolecules',
       'Chemical Kinetics', 'Coordination Compounds',
       'D & F-Block Elements', 'Electrochemistry',
       'Elements Principles & Processes', 'Everyday Life Chemistry',
       'Haloalkanes & Haloarenes', 'P-Block Elements', 'Polymers',
       'Solid State', 'Solutions', 'Surface Chemistry'], dtype=object)

In [7]:
qdf[qdf['domain'] == 'Biomolecules']

,QID,domain,subdomain,question,options,answer,explaination
1563,CH1563,Biomolecules,Carbohydrates - 1,Which of the following carbohydrates does not ...,"['Fructose', 'Glucose', 'Deoxyribose', 'Lactose']",c,"Initially, most of the carbohydrates had the g..."
1564,CH1564,Biomolecules,Carbohydrates - 1,Which of the following carbohydrates does not ...,"['Galactose', 'Sucrose', 'Allolactose', 'Malto...",a,Galactose is a monosaccharide with the formula...
1565,CH1565,Biomolecules,Carbohydrates - 1,The compound with the formula C<sub>2</sub>(H<...,"['carbohydrate', 'carboxylic acid', 'aldehyde'...",b,The compound is acetic acid (CH<sub>3</sub>COO...
1566,CH1566,Biomolecules,Carbohydrates - 1,Identify the correct formula for the carbohydr...,"['C<sub>5</sub>H<sub>10</sub>O<sub>5</sub>', '...",b,Rhamnose is an example of a carbohydrate that ...
1567,CH1567,Biomolecules,Carbohydrates - 1,Which of the following class of compounds is n...,"['Polyamino aldehydes', 'Polyhalo aldehydes', ...",c,Polyamino aldehydes and polyhalo aldehydes do ...
...,...,...,...,...,...,...,...
1701,CH1701,Biomolecules,Hormones,Which of the following does not release steroi...,"['Testes', 'Ovary', 'Adrenal cortex', 'Pancreas']",d,Steroid hormones are produced by adrenal corte...
1702,CH1702,Biomolecules,Hormones,Which hormone controls the balance of water an...,"['Vasopressin', 'Mineralocorticoids', 'Testost...",b,Mineralocorticoids are steroid hormones which ...
1703,CH1703,Biomolecules,Hormones,Lack of which hormone causes Addison’s disease?,"['Glucocorticoids', 'Oxytocin', 'Insulin', 'No...",a,If adrenal cortex does not function properly t...
1704,CH1704,Biomolecules,Hormones,All hormones are proteins.,"['True', 'False']",b,Hormones are compounds having varied chemical ...


In [14]:
q1 = qdf['question'][1166]
q1

'Lucas test was conducted on three compounds A, B and C. Compounds A and B showed turbidity at room temperature, but compound C became turbid only after heating. Which of the compounds has a tertiary structure?'

In [15]:
dr = cr.documentRetriever()
dr.retrieve(q1, 5)
dr.newdf

,domain,subdomain,text,link
122,Organohalides,Preparing Alkyl Halides from Alcohols,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
177,Benzene and Aromaticity,Naming Aromatic Compounds,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
80,Alkenes- Structure and Reactivity,Calculating Degree of Unsaturation,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
286,Amines and Heterocycles,Naming Amines,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
0,Structure and Bonding,Introduction to Organic Chemistry,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...


In [18]:
print(np.unique(cdf['domain']))
cdf[cdf['domain'] == 'Alcohols, Phenols & Ethers']

['Alcohols, Phenols & Ethers' 'Aldehydes, Ketones & Carboxylic Acids'
 'Amines' 'Biomolecules' 'Chemical Kinetics' 'Coordination Compounds'
 'D & F-Block Elements' 'Electrochemistry'
 'Elements Principles & Processes' 'Everyday Life Chemistry'
 'Haloalkanes & Haloarenes' 'P-Block Elements' 'Polymers' 'Solid State'
 'Solutions' 'Surface Chemistry']


,Unnamed: 0,domain,subdomain,content
99,99,"Alcohols, Phenols & Ethers",Alcohols Classification,The classification of compounds makes their st...
100,100,"Alcohols, Phenols & Ethers",Alcohols Nomenclature,(a) Alcohols: The common name of an alcohol is...
101,101,"Alcohols, Phenols & Ethers",Alcohols Nomenclature,(b) Phenols: The simplest hydroxy derivative o...
102,102,"Alcohols, Phenols & Ethers",Functional Group Structure,"In alcohols, the oxygen of the –OH group is at..."
103,103,"Alcohols, Phenols & Ethers",Alcohols & Phenols,11.4.1 Preparation of AlcoholsAlcohols are pre...
104,104,"Alcohols, Phenols & Ethers",Alcohols & Phenols,1. From haloarenesChlorobenzene is fused with ...
105,105,"Alcohols, Phenols & Ethers",Alcohols & Phenols,"Alcohols and phenols consist of two parts, an ..."
106,106,"Alcohols, Phenols & Ethers",Alcohols & Phenols,Alcohols are versatile compounds. They react b...
107,107,"Alcohols, Phenols & Ethers",Commercial Alcohols,Methanol and ethanol are among the two commerc...
108,108,"Alcohols, Phenols & Ethers",Ethers,By dehydration of alcoholsAlcohols undergo deh...


In [27]:
cdf = pd.read_csv('classified.csv')
q2 = cdf['content'][107]

In [28]:
q2

'Methanol and ethanol are among the two commercially importantalcohols.1. MethanolMethanol, CH3OH, also known as ‘wood spirit’, was produced bydestructive distillation of wood. Today, most of the methanol isproduced by catalytic hydrogenation of carbon monoxide at highpressure and temperature and in the presence of ZnO – Cr2O3catalyst.Methanol is a colourless liquid and boils at 337 K. It is highlypoisonous in nature. Ingestion of even small quantities of methanolcan cause blindness and large quantities causes even death. Methanolis used as a solvent in paints, varnishes and chiefly for makingformaldehyde.2. EthanolEthanol, C2H5OH, is obtained commercially by fermentation, theoldest method is from sugars. The sugar in molasses, sugarcaneor fruits such as grapes is converted to glucose and fructose, (bothof which have the formula C6H12O6), in the presence of an enzyme,invertase. Glucose and fructose undergo fermentation in thepresence of another enzyme, zymase, which is found in yeast.I

In [30]:
dr = cr.documentRetriever()
dr.retrieve(q2, 10)
dr.newdf

,domain,subdomain,text,link
198,Alcohols and Phenols,Introduction,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
307,Biomolecules- Carbohydrates,Other Important Carbohydrates,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
298,Biomolecules- Carbohydrates,Classification of Carbohydrates,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
74,An Overview of Organic Reactions,Describing a Reaction- Intermediates,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
305,Biomolecules- Carbohydrates,Disaccharides,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
303,Biomolecules- Carbohydrates,Reactions of Monosaccharides,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
297,Biomolecules- Carbohydrates,Introduction,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
283,Carbonyl Condensation Reactions,Some Biological Carbonyl Condensation Reactions,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
302,Biomolecules- Carbohydrates,Cyclic Structures of Monosaccharides: Anomers,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
321,"Biomolecules- Amino Acids, Peptides, and Proteins",Enzymes and Coenzymes,"Objectives After completing this section, you...",https://chem.libretexts.org/Bookshelves/Organi...
